# Fine-tune cpsam on DIC Microscopy Data

This notebook fine-tunes cellpose's cpsam (ViT-based) model on DIC
(Differential Interference Contrast) keratinocyte recordings.

**Why**: cpsam's default model was trained on phase-contrast/fluorescence
and picks up DIC background texture as false positives. Fine-tuning on
5,290 DIC training pairs (Jesse's VAMPIRE dataset) teaches the ViT
encoder to distinguish DIC cell boundaries from background relief.

**Requirements**: GPU runtime (T4 free tier is sufficient, A100 faster)

**Output**: `cpsam_dic` model file — download and place in
`cellscope/data/models/cpsam_dic`

In [ ]:
# Step 1: Install cellpose 4.x
!pip install cellpose>=4.1.1 tifffile -q

In [ ]:
# Step 1b (optional): keep the Colab session from idle-disconnecting.
# Run this once, then leave the tab open. Auto-clicks the connect button
# every 60 s so a long training run doesn't get killed for inactivity.
from IPython.display import display, Javascript
display(Javascript('''
function ConnectButton() {
    const b = document.querySelector("colab-connect-button");
    if (b) b.click();
}
setInterval(ConnectButton, 60000);
'''))
print('Keep-alive installed.')


In [ ]:
# Step 2: Mount Google Drive (upload training data there first)
from google.colab import drive
drive.mount('/content/drive')

# Training data should be at:
# /content/drive/MyDrive/cellscope_training/dic_splits_v3/train/
# Upload the dic_splits_v3.zip to Drive and unzip:
DRIVE_PATH = '/content/drive/MyDrive/cellscope_training'

import os
if not os.path.exists(f'{DRIVE_PATH}/dic_splits_v3/train'):
    print('Training data not found. Please upload dic_splits_v3.zip to Drive.')
    print('Expected path:', f'{DRIVE_PATH}/dic_splits_v3/train/')
else:
    n_files = len(os.listdir(f'{DRIVE_PATH}/dic_splits_v3/train'))
    print(f'Found {n_files // 2} training pairs')

In [ ]:
# Step 3: Collect file paths (no array loading — saves RAM)
# cellpose's train_seg accepts train_files + train_labels_files and
# loads each pair on demand, so we never hold the whole dataset in RAM.
import glob, os, tifffile

TRAIN_DIR = f'{DRIVE_PATH}/dic_splits_v3/train'

img_files = sorted(glob.glob(f'{TRAIN_DIR}/*_img.tif'))
train_files, train_labels_files = [], []
for f in img_files:
    mf = f.replace('_img.tif', '_masks.tif')
    if os.path.exists(mf):
        train_files.append(f)
        train_labels_files.append(mf)

# Subsample for Colab T4 free (~13 GB RAM): full 2,644 pairs OOMs after
# flow precomputation. 1000 fits comfortably; bump higher with High-RAM.
SUBSAMPLE_TO = 1000     # set to None to use all pairs (needs ≥25 GB RAM)
if SUBSAMPLE_TO and len(train_files) > SUBSAMPLE_TO:
    import random; random.seed(0)
    idx = sorted(random.sample(range(len(train_files)), SUBSAMPLE_TO))
    train_files = [train_files[i] for i in idx]
    train_labels_files = [train_labels_files[i] for i in idx]

# Spot-check one pair so we see shape/dtype without loading the rest.
sample_img = tifffile.imread(train_files[0])
sample_msk = tifffile.imread(train_labels_files[0])
print(f'Found {len(train_files)} pairs')
print(f'Sample image: {sample_img.shape} {sample_img.dtype}, '
      f'mask: {sample_msk.shape} {sample_msk.dtype}, '
      f'max label = {int(sample_msk.max())}')
del sample_img, sample_msk


In [ ]:
# Step 4: Fine-tune cpsam — heartbeat + per-epoch checkpointing
# Why these changes vs the original:
#  • train_files / train_labels_files: streams from Drive, saves RAM
#  • SUBSAMPLE_TO + BATCH_SIZE = 1: keeps memory under T4 free's ~13 GB
#  • save_every = 1: checkpoint to Drive after every epoch
#    so a Colab disconnect doesn't lose work
#  • heartbeat thread: prints elapsed time + RAM + GPU every 60 s
#    so you can see the process is alive between epoch logs
import time, gc, os, threading
import cellpose
from cellpose import models, train

print(f'cellpose version: {cellpose.version}')
gc.collect()
try:
    import torch; torch.cuda.empty_cache()
except Exception:
    pass

model = models.CellposeModel(gpu=True)

N_EPOCHS = 20
LR = 1e-5
BATCH_SIZE = 1
SAVE_EVERY = 1     # checkpoint to Drive after every epoch

OUT_DIR = f'{DRIVE_PATH}/models'
os.makedirs(OUT_DIR, exist_ok=True)

print(f'Training {N_EPOCHS} epochs, lr={LR}, batch={BATCH_SIZE}, '
      f'pairs={len(train_files)}, save_every={SAVE_EVERY}')
print(f'Output: {OUT_DIR}/cpsam_dic (overwritten each epoch)')

# Heartbeat — prints status every 60 s so you can confirm the cell is alive.
def _heartbeat(stop):
    import psutil
    t0 = time.time()
    while not stop.is_set():
        mins = (time.time() - t0) / 60
        ram = psutil.virtual_memory()
        msg = (f'[heartbeat] +{mins:5.1f} min   '
               f'RAM {ram.used/1e9:.1f}/{ram.total/1e9:.1f} GB')
        try:
            import torch
            if torch.cuda.is_available():
                msg += f'   GPU {torch.cuda.memory_allocated()/1e9:.1f} GB'
        except Exception:
            pass
        print(msg, flush=True)
        for _ in range(60):
            if stop.is_set():
                return
            time.sleep(1)

stop = threading.Event()
hb = threading.Thread(target=_heartbeat, args=(stop,), daemon=True)
hb.start()

t0 = time.time()
try:
    new_path, train_losses, test_losses = train.train_seg(
        model.net,
        train_files=train_files,
        train_labels_files=train_labels_files,
        save_path=OUT_DIR,
        n_epochs=N_EPOCHS,
        learning_rate=LR,
        batch_size=BATCH_SIZE,
        save_every=SAVE_EVERY,
        model_name='cpsam_dic',
        min_train_masks=1,
    )
finally:
    stop.set()
    hb.join(timeout=2)

print(f'\nDone in {(time.time()-t0)/60:.1f} minutes')
print(f'Final train loss: {train_losses[-1]:.4f}')
print(f'Model saved: {new_path}')


In [ ]:
# Step 5: Quick validation
VAL_DIR = f'{DRIVE_PATH}/dic_splits_v3/val'

if os.path.exists(VAL_DIR):
    val_imgs = sorted(glob.glob(f'{VAL_DIR}/*_img.tif'))[:20]
    trained = models.CellposeModel(gpu=True, pretrained_model=new_path)
    
    ious = []
    for f in val_imgs:
        img = tifffile.imread(f)
        gt = tifffile.imread(f.replace('_img.tif', '_masks.tif')) > 0
        pred, _, _ = trained.eval(img)
        pred_bool = pred > 0
        inter = np.logical_and(pred_bool, gt).sum()
        union = np.logical_or(pred_bool, gt).sum()
        ious.append(inter / union if union > 0 else 0)
    
    print(f'Validation IoU: {np.mean(ious):.3f} ({len(val_imgs)} frames)')
else:
    print('No validation set found — skip')

In [ ]:
# Step 6: Download the model
from google.colab import files
files.download(new_path)
print('Download complete. Place the model at:')
print('  cellscope/data/models/cpsam_dic')

## After downloading

Place the model file at `cellscope/data/models/cpsam_dic` and it
will be automatically used by the DIC pipeline when the modality
is set to DIC.

To use in Python:
```python
from cellpose import models
model = models.CellposeModel(
    gpu=True,
    pretrained_model='data/models/cpsam_dic')
masks, flows, styles = model.eval(image)
```